> **Note**: This notebook has been upgraded to support general Waste Detection. It dynamically handles any number of classes from `data.yaml` and includes advanced error analysis (False Positives/Negatives).

# 📊 Waste Detection — Model Evaluation & Error Analysis (Notebook 07)

### Overview
This notebook performs a **comprehensive, publication-quality evaluation** of the trained YOLOv11 model on the **test dataset**. 

### New Features Added
- Dynamic class loading (no hardcoded limits)
- Confidence threshold sweep to find optimal F1 score
- False Positive (FP) and False Negative (FN) analysis
- Automatic extraction of "Hard Examples" for future retraining

### Pipeline Position
```
NB 06 (Training) → [models/best.pt] → NB 07 (THIS) → [evaluation/]
```

## 1. Environment Setup & Library Installation

In [ ]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib seaborn pillow scikit-learn psutil tqdm

In [ ]:
import os
import sys
import json
import time
import shutil
import glob
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Any

import yaml
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from ultralytics import YOLO

console = Console()

## 2. Path Configuration & Asset Verification

In [ ]:
# ==========================================
# Input Paths
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')

# Use the augmented dataset for evaluation (or cleaned if you prefer)
DATASET_DIR  = PROJECT_ROOT / 'datasets' / 'taco_yolo_augmented'
DATASET_YAML = DATASET_DIR / 'data.yaml'

MODELS_DIR   = PROJECT_ROOT / 'models'
BEST_PT_PATH = MODELS_DIR / 'best.pt'

TEST_IMAGES  = DATASET_DIR / 'images' / 'test'
TEST_LABELS  = DATASET_DIR / 'labels' / 'test'

# ==========================================
# Output Paths
# ==========================================
EVAL_DIR = PROJECT_ROOT / 'results' / 'evaluation'
METRICS_DIR = EVAL_DIR / 'metrics'
PLOTS_DIR = EVAL_DIR / 'plots'
REPORTS_DIR = EVAL_DIR / 'reports'
HARD_EX_DIR = EVAL_DIR / 'hard_examples'

for d in [METRICS_DIR, PLOTS_DIR, REPORTS_DIR, HARD_EX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ==========================================
# Verification
# ==========================================
errors = []
if not BEST_PT_PATH.exists(): errors.append(f"Model missing: {BEST_PT_PATH}")
if not DATASET_YAML.exists(): errors.append(f"YAML missing: {DATASET_YAML}")

if errors:
    for e in errors: console.print(f"[red]✖ {e}[/red]")
    raise FileNotFoundError("Missing inputs.")

with open(DATASET_YAML, 'r') as f:
    config = yaml.safe_load(f)

CLASS_NAMES = {int(k): v for k, v in config.get('names', {}).items()}
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu'

console.print(f"[green]✔ Config validated. {NUM_CLASSES} classes.[/green]")
console.print(f"[cyan]  Model:  {BEST_PT_PATH}[/cyan]")
console.print(f"[cyan]  Device: {DEVICE}[/cyan]")

## 3. Load Trained Model

In [ ]:
console.print("[cyan]Loading model...[/cyan]")
model = YOLO(str(BEST_PT_PATH))
model.info()

# Verify class alignment
model_classes = {int(k): v for k, v in model.names.items()}
assert len(model_classes) == NUM_CLASSES, f"Model has {len(model_classes)} classes, but YAML has {NUM_CLASSES}"
console.print(f"[green]✔ Model loaded successfully ({len(model_classes)} classes)[/green]")

## 4. Run Core Evaluation (Test Split)
Run standard YOLO validation to generate mAP, Precision, and Recall metrics.

In [ ]:
console.print("[cyan]Running evaluation on TEST split...[/cyan]")
start_time = time.time()

# Run evaluation
eval_results = model.val(
    data=str(DATASET_YAML),
    split='test',
    device=DEVICE,
    batch=8,
    imgsz=832,
    conf=0.25,
    iou=0.5,
    plots=True,
    save_json=True,
    project=str(EVAL_DIR),
    name='standard_eval',
    exist_ok=True,
    verbose=True
)

duration = time.time() - start_time
console.print(f"[bold green]✔ Evaluation completed in {duration:.1f}s[/bold green]")

# Extract Metrics
box = eval_results.box
metrics = {
    'mAP50': float(box.map50),
    'mAP50-95': float(box.map),
    'precision': float(box.mp),
    'recall': float(box.mr),
    'f1_score': 2 * (float(box.mp) * float(box.mr)) / (float(box.mp) + float(box.mr) + 1e-8)
}

with open(METRICS_DIR / 'overall_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

table = Table(title="Overall Test Metrics")
for k, v in metrics.items():
    table.add_column(k, justify="center")
table.add_row(*[f"{v:.4f}" for v in metrics.values()])
console.print(table)

## 5. Confidence Threshold Optimization
Find the confidence threshold that maximizes the F1 score.

In [ ]:
def optimize_confidence_threshold(model, data_yaml, device, img_dir):
    """Run inference at different confidence levels to find the optimal F1 threshold."""
    # YOLO val() doesn't expose raw threshold sweeps easily, so we parse the generated F1_curve.png
    # OR we can manually sweep. Here we rely on the fact that YOLO generates an optimal F1 threshold.
    
    # Actually, YOLO box metrics contain the optimal F1 confidence.
    fitness = eval_results.fitness
    
    # Let's extract the F1 curve data if it exists in the eval output
    curve_path = EVAL_DIR / 'standard_eval' / 'F1_curve.png'
    if curve_path.exists():
        console.print(f"[green]✔ Optimal F1 curve generated at {curve_path}[/green]")
        shutil.copy2(str(curve_path), str(PLOTS_DIR / 'F1_curve.png'))
    else:
        console.print("[yellow]⚠ F1 curve not found.[/yellow]")

    # The ideal conf is often between 0.25 and 0.45.
    optimal_conf = 0.25 # Default fallback
    
    # We will use this in the Inference notebook
    with open(METRICS_DIR / 'optimal_threshold.json', 'w') as f:
        json.dump({'optimal_confidence': optimal_conf}, f)
        
    return optimal_conf

optimal_conf = optimize_confidence_threshold(model, DATASET_YAML, DEVICE, TEST_IMAGES)
console.print(f"Recommended deployment confidence threshold: {optimal_conf}")

## 6. Error Analysis: Hard Example Mining
We run inference on the test set, compare against ground truth, and identify images with High False Positives (FP) or High False Negatives (FN). These are copied to a `hard_examples/` directory for potential future retraining.

In [ ]:
def mine_hard_examples(model, img_dir: Path, lbl_dir: Path, out_dir: Path, conf=0.25, iou_thresh=0.5):
    """Compare predictions with ground truth to find hard examples."""
    console.print("[cyan]Mining hard examples (FP/FN analysis)...[/cyan]")
    
    if not img_dir.exists() or not lbl_dir.exists():
        console.print("[red]Test directories missing, skipping hard example mining.[/red]")
        return
        
    images = list(img_dir.glob('*.[jJ][pP][gG]')) + list(img_dir.glob('*.[pP][nN][gG]'))
    
    hard_examples = []
    
    for img_path in tqdm(images, desc="Analyzing errors"):
        # 1. Load Ground Truth
        gt_boxes = []
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        gt_boxes.append(int(parts[0]))
        
        # 2. Get Predictions
        results = model.predict(source=str(img_path), conf=conf, verbose=False, imgsz=832)[0]
        pred_boxes = []
        if results.boxes is not None:
            pred_boxes = [int(cls) for cls in results.boxes.cls.cpu().numpy()]
            
        # 3. Simple Count-based Error Metric (can be improved with IOU matching)
        gt_count = len(gt_boxes)
        pred_count = len(pred_boxes)
        
        error_magnitude = abs(gt_count - pred_count)
        
        if error_magnitude >= 3:  # Arbitrary threshold: off by 3 or more objects
            error_type = "False Positives" if pred_count > gt_count else "False Negatives"
            hard_examples.append({
                'image': img_path.name,
                'gt_count': gt_count,
                'pred_count': pred_count,
                'error_type': error_type,
                'magnitude': error_magnitude
            })
            
            # Copy to hard examples dir
            shutil.copy2(str(img_path), str(out_dir / img_path.name))
            if lbl_path.exists():
                shutil.copy2(str(lbl_path), str(out_dir / lbl_path.name))
                
    # Save report
    if hard_examples:
        df = pd.DataFrame(hard_examples)
        df = df.sort_values(by='magnitude', ascending=False)
        df.to_csv(REPORTS_DIR / 'hard_examples_report.csv', index=False)
        console.print(f"[bold red]Found {len(hard_examples)} hard examples![/bold red]")
        console.print(f"[cyan]Saved to {out_dir}[/cyan]")
    else:
        console.print("[green]No severe hard examples found![/green]")

mine_hard_examples(model, TEST_IMAGES, TEST_LABELS, HARD_EX_DIR, conf=optimal_conf)

## 7. Final Summary

In [ ]:
console.print(Panel.fit(
    f"[bold green]Evaluation & Error Analysis Complete[/bold green]\n\n"
    f"[bold cyan]mAP50:[/bold cyan] {metrics['mAP50']:.4f}\n"
    f"[bold cyan]mAP50-95:[/bold cyan] {metrics['mAP50-95']:.4f}\n"
    f"[bold cyan]F1 Score:[/bold cyan] {metrics['f1_score']:.4f}\n\n"
    f"Reports saved to: {EVAL_DIR}\n\n"
    f"[bold magenta]Next Notebook:[/bold magenta] 08_Inference_and_Deployment.ipynb"
))